# 제출용 추론 노트북

**이 노트북을 동결(Save Version)하여 제출합니다.**

이 노트북은 **학습된 모델을 불러와 예측만** 수행합니다.

---

## 지켜야 할 규칙

**1. 날짜를 하드코딩하지 마세요.** 예측 기간은 반드시 `PRED_DATES`를 참조해야 합니다.  
운영진은 `PRED_START`, `PRED_END`만 바꿔 실행합니다. 날짜가 코드에 박혀 있으면 채점이 불가능합니다.

**2. 예측 대상 시각은 매일 14:00 KST입니다.**  
설정 셀에서 `PRED_DATES`의 각 원소를 **14:00 KST가 포함된 `datetime`** 으로 만들어 줍니다.


위성 API의 `date` 파라미터는 **UTC 기준**이므로 설정 셀의 `to_api_datetime()`을 사용하세요.  
예: `2026-08-24 14:00 KST → 202608240500 (UTC)`

**3. 마지막에 `pred` DataFrame을 만드세요.** 형식은 아래 설명을 참고하세요.

## 사전 준비

1. 학습된 모델을 파일로 저장 → **캐글 Dataset으로 업로드** (형식 자유)
   - Public으로 설정하거나 운영진 계정에 공유
2. 우측 **Add Input** 으로 연결
   - 대회 데이터셋 (`station_list.csv`)
3. **Session options → Internet: On**

> `station_list.csv` 는 **어떤 지점을 채점하는지만** 알려줍니다.

## 1. 설정 — 수정하지 마세요

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════╗
# ║  ★ 운영진 수정 구역 — 채점 시 아래 3줄만 교체합니다 ★               ║
# ╚═══════════════════════════════════════════════════════════════════╝
API_KEY    = ""                # 기상청 API Hub 인증키
PRED_START = "20260624"        # 예측 시작일 (YYYYMMDD)
PRED_END   = "20260630"        # 예측 종료일 (YYYYMMDD)
# ═══════════════════════════════════════════════════════════════════════

import os, sys, glob
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import requests

if not API_KEY:
    sys.exit("API_KEY를 입력하세요. (Kaggle Secrets는 사용할 수 없습니다)")

# ── 평가 기준 시각: 수정하지 마세요 ─────────────────────────────
EVAL_HOUR = 14
EVAL_MINUTE = 0

# 예측 대상 일시
# 중요: PRED_DATES의 각 원소는 '날짜만'이 아니라 매일 14:00 KST의 datetime 입니다.
_start_dt = datetime.strptime(PRED_START, "%Y%m%d").replace(
    hour=EVAL_HOUR, minute=EVAL_MINUTE, second=0, microsecond=0
)
_last_dt = datetime.strptime(PRED_END, "%Y%m%d").replace(
    hour=EVAL_HOUR, minute=EVAL_MINUTE, second=0, microsecond=0
)

if _start_dt > _last_dt:
    sys.exit("PRED_START는 PRED_END보다 늦을 수 없습니다.")

PRED_DATES = []
_dt = _start_dt
while _dt <= _last_dt:
    PRED_DATES.append(_dt)
    _dt += timedelta(days=1)

# 방어적 검증: 예측 대상 시각이 실수로 00시 등으로 바뀌는 것을 차단
if any(
    (pd.Timestamp(d).hour != EVAL_HOUR) or
    (pd.Timestamp(d).minute != EVAL_MINUTE)
    for d in PRED_DATES
):
    sys.exit("PRED_DATES 생성 오류: 모든 예측 대상 시각은 14:00 KST여야 합니다.")

def _as_kst(dt):
    """naive datetime은 KST로 해석하고, timezone-aware 값은 KST로 변환합니다."""
    ts = pd.Timestamp(dt)
    if ts.tzinfo is None:
        return ts.tz_localize("Asia/Seoul")
    return ts.tz_convert("Asia/Seoul")

# 위성 API 요청용 시각 문자열 생성 함수
def to_api_datetime(obs_dt, target_dt=None):

    if target_dt is None:
        target_dt = obs_dt

    obs = _as_kst(obs_dt)
    target = _as_kst(target_dt)

    
    if target.hour != EVAL_HOUR or target.minute != EVAL_MINUTE:
        raise ValueError(
            f"잘못된 예측 대상 시각: {target}. "
            f"target_dt는 {EVAL_HOUR:02d}:{EVAL_MINUTE:02d} KST여야 합니다."
        )

   
    if obs > target:
        raise ValueError(
            f"미래 위성영상 사용 금지: obs_dt={obs}, target_dt={target}. "
            "추론에는 예측 대상 시각까지 관측된 위성영상만 사용할 수 있습니다."
        )

    # KMA GK-2A LE1B API date는 UTC 기준
    return obs.tz_convert("UTC").strftime("%Y%m%d%H%M")


from urllib.parse import urlparse, parse_qs

API_AUDIT_LOG = []

if not hasattr(requests.sessions.Session, "_competition_original_request"):
    requests.sessions.Session._competition_original_request = requests.sessions.Session.request

_ORIGINAL_REQUEST = requests.sessions.Session._competition_original_request

def _audit_request(self, method, url, **kwargs):
    try:
        parsed = urlparse(str(url))
        host = parsed.netloc

        if "apihub.kma.go.kr" in host:
            query = parse_qs(parsed.query, keep_blank_values=True)

            params = kwargs.get("params")
            if isinstance(params, dict):
                for k, v in params.items():
                    if k == "authKey":
                        continue
                    if isinstance(v, (list, tuple)):
                        query[k] = [str(x) for x in v]
                    else:
                        query[k] = [str(v)]

            def _first(name):
                vals = query.get(name, [])
                return vals[0] if vals else None

            API_AUDIT_LOG.append({
                "method": str(method).upper(),
                "host": host,
                "path": parsed.path,
                "date": _first("date"),
                "sDate": _first("sDate"),
                "eDate": _first("eDate"),
            })
    except Exception as e:
        print(f"[경고] API 로그 기록 실패: {type(e).__name__}: {e}")

    return _ORIGINAL_REQUEST(self, method, url, **kwargs)

requests.sessions.Session.request = _audit_request

# 평가 대상 지점 ── station_list.csv 에 정의된 공식 96개
# glob 반환 순서에 의존하지 않고, STN_ID 96개 후보를 검증해서 선택합니다.
_hits = sorted(glob.glob("/kaggle/input/**/station_list.csv", recursive=True))
if not _hits:
    sys.exit("station_list.csv 를 찾을 수 없습니다. Add Input을 확인하세요.")

_valid_station_files = []
for _p in _hits:
    try:
        _tmp = pd.read_csv(_p)
        if "STN_ID" not in _tmp.columns:
            continue
        _ids = tuple(sorted(_tmp["STN_ID"].dropna().astype(int).unique().tolist()))
        if len(_ids) == 96:
            _valid_station_files.append((_p, _ids))
    except Exception:
        continue

if not _valid_station_files:
    sys.exit(
        "STN_ID 96개를 가진 station_list.csv를 찾지 못했습니다. "
        "공식 대회 데이터셋 연결을 확인하세요."
    )

_station_id_sets = {ids for _, ids in _valid_station_files}
if len(_station_id_sets) > 1:
    _paths = [p for p, _ in _valid_station_files]
    sys.exit(
        "서로 다른 96지점 station_list.csv가 여러 개 발견되었습니다. "
        "공식 대회 데이터셋만 남기거나 중복 파일명을 변경하세요.\n"
        + "\n".join(_paths)
    )

_station_path, _station_ids = _valid_station_files[0]
STATIONS = list(_station_ids)

if len(_valid_station_files) > 1:
    print(
        f"[경고] 동일한 96지점 station_list.csv가 {len(_valid_station_files)}개 발견되었습니다. "
        f"다음 파일을 사용합니다: {_station_path}"
    )
else:
    print(f"station_list.csv: {_station_path}")

print(f"예측 기간 : {PRED_START} ~ {PRED_END}  ({len(PRED_DATES)}일)")
print(f"예측 대상 시각 : 매일 {EVAL_HOUR:02d}:{EVAL_MINUTE:02d} KST")
print(f"평가 지점 : {len(STATIONS)}개")
print(
    "API 시각 예시 : "
    f"{PRED_DATES[0].strftime('%Y-%m-%d %H:%M')} KST -> "
    f"{to_api_datetime(PRED_DATES[0])} UTC"
)

## 2. 자유 구현 ★ 여기만 채우세요 ★

### `pred` 반환 형식

| 컬럼 | 내용 |
|---|---|
| `Date` | 정수 `YYYYMMDD` |
| `STN_ID` | 정수 지점번호 |
| `TA` | 기온 예측값 (°C) |
| `HM` | 습도 예측값 (%) |

`PRED_DATES` × `STATIONS` 의 **모든 조합**이 정확히 한 번씩 있어야 하고, 결측이 없어야 합니다.

### 평가/위성 시각 규칙 — 중요

- **예측 대상 시각은 매일 14:00 KST**입니다.
- 설정 셀의 `PRED_DATES`에는 이미 각 평가일 14:00 KST가 들어 있습니다.
- 위성 입력은 **해당 `target_dt`까지 관측된 자료**를 사용할 수 있습니다.
- GK-2A LE1B API의 `date`는 **UTC 기준**입니다.
- `to_api_datetime(obs_dt, target_dt)`를 사용하면 KST 기준 규칙을 확인한 뒤 UTC `YYYYMMDDHHMM`으로 변환합니다.
- **naive `obs_dt`는 KST로 해석됩니다. 이미 UTC인 값을 timezone 없이 넣지 마세요. UTC 값을 사용할 경우 timezone-aware UTC Timestamp로 전달하세요.**
- 예: `2026-08-24 14:00 KST → 202608240500 UTC`
- 시간 피처는 **학습 때 정의한 시간 기준과 동일하게** 만드세요. KST 기준 학습이면 14, UTC 기준 학습이면 5입니다.

### 유의사항

- **학습 코드를 넣지 마세요.** 추론 전용입니다. 실행 시간이 비정상적으로 길면 재학습으로 간주될 수 있습니다.
- **무작위성을 제거하세요.** 같은 입력에 항상 같은 출력이 나와야 합니다.
- **학습 때와 동일한 방식으로 피처를 만드세요.**
- 평가기간의 위성자료를 추론 입력으로 사용할 수 있지만, 이를 이용해 모델을 추가 학습·파인튜닝하면 안 됩니다.
- API 오류를 `except Exception:`으로 숨기지 말고, 디버깅할 때는 예외 종류와 메시지를 확인하세요.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  ↓↓↓ 자유 구현 영역 ↓↓↓
#
#  설계 원칙
#  1) 입력은 GK-2A LE1B 위성영상과 정적 지점정보(위경도·고도)뿐이다.
#     ASOS 는 학습 라벨로만 쓰였고, 추론에는 어떤 형태로도 들어가지 않는다
#     (기후 평년값·지점별 과거 평균기온 포함 — 운영진 답변에 따름).
#     위성이 결측이면 모델이 위경도·고도·연중일만으로 예측한다.
#     (규정: "위성 입력이 결측되거나 없는 경우에도 예측값이 존재해야 함")
#  2) 인과성 준수 — 대상 시각 14:00 KST = 05:00 UTC 까지의 영상만 쓴다.
#     당일 06 UTC 이후는 정답 시각 이후라 요청조차 하지 않는다.
#  3) 무작위성 없음 — 같은 입력이면 항상 같은 출력.
# ═══════════════════════════════════════════════════════════════════
import glob
import io
import pickle
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import requests

# GK2A LE1B 파일은 확장자가 .nc 지만 실제 형식은 HDF5 다 (매직바이트 \x89HDF).
# 그래서 netCDF4 없이 h5py 만으로 그대로 읽을 수 있고, h5py 는 캐글 기본
# 환경에 들어 있다. 두 경로가 같은 배열을 준다는 것은 확인했다.
# 추가 설치에 기대지 않는다 — pip 는 네트워크가 막히면 그대로 실패한다.
import h5py

print(f"h5py {h5py.__version__} — GK2A(HDF5) 판독 준비 완료")

# ── 학습 산출물 로드 ──────────────────────────────────────────────
_hits = glob.glob("/kaggle/input/**/model.pkl", recursive=True)
if not _hits:
    sys.exit("model.pkl 을 찾을 수 없습니다. Add Input 을 확인하세요.")
BUNDLE = pickle.load(open(_hits[0], "rb"))

MODELS   = BUNDLE["models"]
LINEARS  = BUNDLE.get("linears", {})   # 외삽 담당 선형 성분
FEATURES = BUNDLE["features"]
GBM_FEATURES = BUNDLE.get("gbm_features", FEATURES)
# 연도 피처를 타깃마다 다르게 쓰므로 입력 집합도 타깃별로 갈린다
FEAT_BY_T = BUNDLE.get("features_by_target", {})
GBMF_BY_T = BUNDLE.get("gbm_features_by_target", {})
CHANNELS = BUNDLE["channels"]
WIN_STATS = BUNDLE["win_stats"]
STN_META = BUNDLE["stations"]
ROWS     = BUNDLE["pixel_rows"].astype(int)
COLS     = BUNDLE["pixel_cols"].astype(int)
H_TODAY  = BUNDLE["hours_utc"]       # [0, 2, 4, 5]  당일 05 UTC 이하
H_PREV   = BUNDLE["hours_utc_prev"]  # [18, 21]      전날
HALF     = 4                          # 9x9 창

# 번들의 지점 순서와 채점 지점 순서를 맞춘다
_order = {s: i for i, s in enumerate(STN_META["STN"].astype(int))}
_idx = np.array([_order[s] for s in STATIONS])
ROWS, COLS = ROWS[_idx], COLS[_idx]

print(f"모델 로드 완료 — 채널 {CHANNELS}, 피처 {len(FEATURES)}개")

# ── 위성 다운로드 (실패해도 죽지 않고, 오래 끌지도 않는다) ────────
#
# 기상청 API 허브는 실제로 몇 시간씩 죽는 일이 있다. 그때 재시도에 매달리면
# 노트북이 런타임 한도를 넘겨 채점 자체가 불가능해진다. 그래서
#   - 요청당 타임아웃을 짧게 잡고
#   - 전체 다운로드에 시간 예산을 두며
#   - 초반이 연속 실패하면 서버가 죽은 것으로 보고 즉시 포기한다.
# 포기해도 지점정보·연중일만으로 예측은 그대로 나간다.
HUB = "https://apihub.kma.go.kr/api/typ05/api/GK2A/LE1B"

# ── 타임아웃 설계 (실측 기반) ─────────────────────────────────────
# 이 API 는 실패할 때 HTTP 오류를 주지 않고 연결이 그대로 멈춘다.
# 성공하면 0.3초, 실패하면 영영 응답이 없다. 따라서 오래 기다리는 것은
# 순수한 낭비이고, 짧게 끊고 새 연결로 다시 붙는 편이 빠르면서 성공률도 높다.
#
#   실측 (10장 기준)
#     타임아웃  5s x 8회  ->  10/10 성공, 평균 15.2초
#     타임아웃  8s x 6회  ->   8/10 성공, 평균 24.4초
#     타임아웃 15s x 4회  ->   8/10 성공, 평균 34.0초
REQ_TIMEOUT = 5           # 초 — 정상 응답은 0.3초면 온다
MAX_TRIES = 10            # 무응답은 재연결로만 뚫린다
RETRY_SLEEP = 0.3         # 초 — 대기가 아니라 재시도가 이득이므로 짧게

BUDGET_SEC = 2400         # 초 — 다운로드 전체 예산 (40분)
BREAKER_N = 60            # 연속 실패가 이만큼이면 서버 장애로 판단
MAX_PASSES = 5            # 남은 것을 다시 훑는 최대 횟수

WORKERS = 3
MIN_INTERVAL = 0.2

_last = [0.0]
_t0 = time.monotonic()
_err_kinds = {}
_consec_fail = [0]
_dead = [False]


def _fetch(channel, stamp, tries=MAX_TRIES):
    """한 장 다운로드. 실패하면 None. 예산 초과·차단기 작동 시 즉시 None."""
    if _dead[0] or time.monotonic() - _t0 > BUDGET_SEC:
        return None
    url = f"{HUB}/{channel}/KO/data"
    for _ in range(tries):
        if _dead[0] or time.monotonic() - _t0 > BUDGET_SEC:
            return None
        try:
            wait = MIN_INTERVAL - (time.monotonic() - _last[0])
            if wait > 0:
                time.sleep(wait)
            _last[0] = time.monotonic()
            r = requests.get(url, params={"date": stamp, "authKey": API_KEY},
                             timeout=REQ_TIMEOUT)
            if r.status_code == 200 and len(r.content) > 100_000:
                _consec_fail[0] = 0   # 하나라도 성공하면 장애 판단을 되돌린다
                return r.content
            if len(r.content) < 500 and b'"status"' in r.content:
                # 권한 없음·한도 초과는 재시도해도 소용없다
                break
        except Exception as e:
            # 조용히 삼키지 않는다. 종류별로 세어 마지막에 보고한다.
            _err_kinds[type(e).__name__] = _err_kinds.get(type(e).__name__, 0) + 1
        time.sleep(RETRY_SLEEP)

    _consec_fail[0] += 1
    if _consec_fail[0] >= BREAKER_N and not _dead[0]:
        _dead[0] = True
        print(f"[차단기] 연속 {BREAKER_N}회 실패 — API 장애로 판단, "
              f"위성 수집을 중단하고 지점정보만으로 진행합니다.")
    return None


def _read_image(raw):
    """GK2A 파일 바이트 -> 화소 배열. 디스크에 쓰지 않고 메모리에서 연다."""
    try:
        with h5py.File(io.BytesIO(raw), "r") as f:
            return np.asarray(f["image_pixel_values"][:])
    except Exception as e:
        _err_kinds[f"read:{type(e).__name__}"] = (
            _err_kinds.get(f"read:{type(e).__name__}", 0) + 1)
        return None


def _windows(img):
    """지점별 9x9 창 통계. 격자 밖이면 NaN."""
    n, w = len(ROWS), 2 * HALF + 1
    out = np.full((n, w, w), np.nan, dtype=np.float32)
    H, W = img.shape
    for i in range(n):
        r0, c0 = ROWS[i] - HALF, COLS[i] - HALF
        if r0 >= 0 and c0 >= 0 and r0 + w <= H and c0 + w <= W:
            out[i] = img[r0:r0 + w, c0:c0 + w]
    flat = out.reshape(n, -1)
    c = w // 2
    with np.errstate(all="ignore"):
        return {"c": out[:, c, c],
                "m9": np.nanmean(flat, axis=1),
                "sd9": np.nanstd(flat, axis=1),
                "min9": np.nanmin(flat, axis=1),
                "max9": np.nanmax(flat, axis=1)}


# ── 관측 시각 (KST) ──────────────────────────────────────────────
# 학습 때 쓴 시각은 UTC 기준이지만, 대회 헬퍼 to_api_datetime() 은 KST 를
# 받아 UTC 로 바꿔준다. KST 로 환산하면 여섯 시각 모두 '대상일 당일' 이다.
#
#   학습 태그   UTC            KST(대상일)
#   h00         00             09:00
#   h02         02             11:00
#   h04         04             13:00
#   h05         05             14:00   <- 대상 시각과 동일
#   p18         전날 18        03:00
#   p21         전날 21        06:00
#
# 피처 이름(태그)은 학습 때와 똑같이 유지해야 하므로 그대로 쓴다.
_OBS_KST = ([(f"h{h:02d}", h + 9) for h in H_TODAY]
            + [(f"p{h:02d}", h + 9 - 24) for h in H_PREV])

_jobs = []
for target_dt in PRED_DATES:
    day = pd.Timestamp(target_dt).normalize()
    for tag, kst_hour in _OBS_KST:
        obs_dt = day + pd.Timedelta(hours=kst_hour)
        # 헬퍼가 obs > target 이면 예외를 던져 인과성을 보증한다
        api_date = to_api_datetime(obs_dt, target_dt)
        for ch in CHANNELS:
            _jobs.append((target_dt, api_date, tag, ch))

print(f"위성 요청 {len(_jobs)}건 ({len(PRED_DATES)}일 x "
      f"{len(_OBS_KST)}시각 x {len(CHANNELS)}채널)")
print(f"  관측 시각(KST): {[h for _, h in _OBS_KST]}시  -> 대상 14시 이하 확인됨")

def _download(jobs):
    """목록을 받아 성공한 것만 _stats 에 채우고, 실패 목록을 돌려준다."""
    left = []
    with ThreadPoolExecutor(max_workers=WORKERS) as ex:
        futs = {ex.submit(_fetch, ch, stamp): (d, stamp, tag, ch)
                for (d, stamp, tag, ch) in jobs}
        for fut in as_completed(futs):
            d, stamp, tag, ch = futs[fut]
            raw = fut.result()
            img = _read_image(raw) if raw is not None else None
            if img is None:
                left.append((d, stamp, tag, ch))
            else:
                _stats[(d, tag, ch)] = _windows(img)
    return left


_stats = {}   # (date, tag, channel) -> {stat: array}
_left = _download(_jobs)
print(f"1차: 성공 {len(_jobs)-len(_left)} / 실패 {len(_left)}")

# 남은 것을 여러 번 더 훑는다. 실패는 대부분 일시적 무응답이라 다시
# 붙으면 뚫린다. 예산이 남아 있고 진전이 있는 동안만 반복하며,
# 차단기가 걸렸다면 서버가 정말 죽은 것이므로 멈춘다.
_round = 1
while (_left and not _dead[0] and _round < MAX_PASSES
       and time.monotonic() - _t0 < BUDGET_SEC):
    _round += 1
    _before = len(_left)
    time.sleep(3)
    _left = _download(_left)
    print(f"{_round}차 재시도 후: 실패 {len(_left)}")
    if len(_left) == _before:      # 더 이상 줄지 않으면 그만
        break
print(f"다운로드 최종 성공 {len(_jobs)-len(_left)} / {len(_jobs)}")
if _err_kinds:
    print("  발생한 예외:", ", ".join(f"{k} x{v}" for k, v in
                                   sorted(_err_kinds.items(), key=lambda x: -x[1])))

# ── 피처 조립 (학습 때와 정확히 같은 이름·정의) ───────────────────
_TAGS = [f"h{h:02d}" for h in H_TODAY] + [f"p{h:02d}" for h in H_PREV]
_nan = lambda: np.full(len(STATIONS), np.nan)
rows = []
for d in PRED_DATES:
    rec = {"STN": np.asarray(STATIONS)}
    for ch in CHANNELS:
        for tag in _TAGS:
            got = _stats.get((d, tag, ch))
            for s in WIN_STATS:
                rec[f"{ch}_{s}_{tag}"] = got[s] if got else _nan()
        for s in WIN_STATS:
            base = f"{ch}_{s}"
            rec[f"{base}_d1h"] = rec[f"{base}_h05"] - rec[f"{base}_h04"]
            rec[f"{base}_dnt"] = rec[f"{base}_h05"] - rec[f"{base}_p21"]
    # 분리대기창 보정항 (KMA LST 알고리즘의 T13 - T15)
    if "IR105" in CHANNELS and "IR123" in CHANNELS:
        for tag in _TAGS:
            for s in ("c", "m9"):
                a, b = f"IR105_{s}_{tag}", f"IR123_{s}_{tag}"
                if a in rec and b in rec:
                    rec[f"SWD_{s}_{tag}"] = rec[a] - rec[b]
            # 청천 화소 기준 (DN 최솟값 = 물리적으로 가장 뜨거운 화소)
            a, b = f"IR105_min9_{tag}", f"IR123_min9_{tag}"
            if a in rec and b in rec:
                rec[f"SWDmin_{tag}"] = rec[a] - rec[b]
            # 구름량 대리 (창 안의 최대-최소)
            a, b = f"IR105_max9_{tag}", f"IR105_min9_{tag}"
            if a in rec and b in rec:
                rec[f"CLD_{tag}"] = rec[a] - rec[b]
    f = pd.DataFrame(rec)
    f["date"] = pd.Timestamp(d).normalize()   # 피처는 날짜 단위
    rows.append(f)

X = pd.concat(rows, ignore_index=True)
X = X.merge(STN_META[["STN", "LAT", "LON", "HT"]], on="STN", how="left")
X["DOY"] = X["date"].dt.dayofyear
X["DOY_SIN"] = np.sin(2 * np.pi * X["DOY"] / 365.25)
X["DOY_COS"] = np.cos(2 * np.pi * X["DOY"] / 365.25)
X["YEAR"] = X["date"].dt.year.astype(float)   # 온난화 추세 외삽용 (규칙 2.1③)

# 전국 대비 편차 — 그날 한반도 전체 대비 이 지점이 얼마나 특이한가.
# 학습 때와 같은 정의: 같은 날짜 96개 지점의 평균을 빼준다.
_nat_keys = sorted({c[4:] for c in
                    set(FEAT_BY_T.get("TA14", [])) | set(GBMF_BY_T.get("TA14", []))
                    if c.startswith("ANO_")})
if _nat_keys:
    _have = [c for c in _nat_keys if c in X.columns]
    _mean = X.groupby("date")[_have].transform("mean")
    X = pd.concat([X, pd.DataFrame({f"ANO_{c}": X[c] - _mean[c] for c in _have},
                                   index=X.index)], axis=1)

# 최근접 8개 지점의 위성 평균 (대상 시각). 학습 때와 같은 정의.
_nb_need = sorted({c[3:] for c in set(FEAT_BY_T.get("TA14", []))
                   if c.startswith("NB_")})
if _nb_need:
    _s = STN_META.set_index("STN")
    _la = _s.loc[STATIONS, "LAT"].to_numpy(); _lo = _s.loc[STATIONS, "LON"].to_numpy()
    _d = np.sqrt(((_la[:, None] - _la) * 111) ** 2
                 + ((_lo[:, None] - _lo) * 111 * np.cos(np.radians(_la[:, None]))) ** 2)
    np.fill_diagonal(_d, 1e9)
    _near = {STATIONS[i]: [STATIONS[j] for j in np.argsort(_d[i])[:8]]
             for i in range(len(STATIONS))}
    for _c in _nb_need:
        if _c not in X.columns:
            X[f"NB_{_c}"] = np.nan; continue
        _p = X.pivot_table(index="date", columns="STN", values=_c)
        _m = pd.DataFrame({s_: _p[[x for x in _near[s_] if x in _p.columns]].mean(axis=1)
                           for s_ in STATIONS if s_ in _p.columns})
        _st = _m.stack().rename(f"NB_{_c}").reset_index()
        _st.columns = ["date", "STN", f"NB_{_c}"]
        X = X.merge(_st, on=["date", "STN"], how="left")

    # 방향별 이웃 (동/서/남/북 각 3개) — 학습 때와 같은 정의
    _dy = (_la[:, None] - _la) * 111
    _dx = (_lo[:, None] - _lo) * 111 * np.cos(np.radians(_la[:, None]))
    _sect = {"W": (_dx < 0) & (np.abs(_dx) > np.abs(_dy)),
             "E": (_dx > 0) & (np.abs(_dx) > np.abs(_dy)),
             "S": (_dy < 0) & (np.abs(_dy) >= np.abs(_dx)),
             "N": (_dy > 0) & (np.abs(_dy) >= np.abs(_dx))}
    for _tag, _mask in _sect.items():
        _grp = {}
        for _i in range(len(STATIONS)):
            _cand = np.where(_mask[_i])[0]
            _grp[STATIONS[_i]] = ([STATIONS[j] for j in _cand[np.argsort(_d[_i, _cand])[:3]]]
                                  if len(_cand) else [])
        for _c in _nb_need:
            if _c not in X.columns:
                X[f"D{_tag}_{_c}"] = np.nan; continue
            _p = X.pivot_table(index="date", columns="STN", values=_c)
            _m = pd.DataFrame({
                s_: (_p[[x for x in _grp[s_] if x in _p.columns]].mean(axis=1)
                     if _grp.get(s_) else np.nan)
                for s_ in STATIONS if s_ in _p.columns})
            _st = _m.stack().rename(f"D{_tag}_{_c}").reset_index()
            _st.columns = ["date", "STN", f"D{_tag}_{_c}"]
            X = X.merge(_st, on=["date", "STN"], how="left")
    for _c in _nb_need:
        if f"DE_{_c}" in X and f"DW_{_c}" in X:
            X[f"dEW_{_c}"] = X[f"DE_{_c}"] - X[f"DW_{_c}"]
        if f"DN_{_c}" in X and f"DS_{_c}" in X:
            X[f"dNS_{_c}"] = X[f"DS_{_c}"] - X[f"DN_{_c}"]

# 위성 천정각의 경로 길이 효과 (물리식에 sec(theta)-1 로 들어간다).
# GK-2A 는 정지위성이라 지점별 상수이며 번들에 실린 값을 그대로 쓴다.
if "SEC_SATZEN" in FEATURES:
    _sz = BUNDLE["sat_zenith"]
    X = X.merge(pd.DataFrame({"STN": STN_META["STN"].astype(int).to_numpy(),
                              "SEC_SATZEN": 1.0 / np.cos(np.radians(_sz)) - 1.0}),
                on="STN", how="left")

# 지점번호를 학습 때와 같은 범주 목록으로 맞춘다 (순서가 다르면 코드가 어긋난다)
if "STN_CAT" in GBM_FEATURES:
    X["STN_CAT"] = pd.Categorical(X["STN"].astype(int),
                                  categories=BUNDLE["stn_categories"])

_need = set(FEATURES) | set(GBM_FEATURES)
for _v in list(FEAT_BY_T.values()) + list(GBMF_BY_T.values()):
    _need |= set(_v)
for c in _need:                              # 학습 때 없던 컬럼 방어
    if c not in X.columns:
        X[c] = np.nan

# ── 예측 ──────────────────────────────────────────────────────────
# 시드별 모델의 평균을 쓴다 (학습 때와 동일).
def _predict(key):
    lf = FEAT_BY_T.get(key, FEATURES)
    gf = GBMF_BY_T.get(key, GBM_FEATURES)
    Z = X[lf].astype(np.float64)                 # 선형 성분 입력
    G = X[gf]                                    # GBM 입력 (지점번호 포함)
    ms = MODELS[key]
    ms = ms if isinstance(ms, list) else [ms]
    p = np.mean([m.predict(G) for m in ms], axis=0)
    if key in LINEARS:                 # 잔차 학습이므로 선형분을 되더한다
        p = p + LINEARS[key].predict(Z)
    return p

ta = _predict("TA14")
hm = _predict("HM14")

# 위성이 결측인 행도 모델이 그대로 처리한다. LightGBM 은 결측을 학습 때
# 배운 방향으로 보내므로, 위성이 전혀 없으면 위경도·고도·연중일만으로
# 예측이 나온다. 기후 평년값 같은 ASOS 파생 자료로 대체하지 않는다
# (운영진: ASOS 는 학습 라벨로만 사용 가능).
_sat_cols = [c for c in FEATURES if c.split("_")[0] in CHANNELS]
_no_sat = int(X[_sat_cols].isna().all(axis=1).sum())
if _no_sat:
    print(f"위성 전무 {_no_sat}행 — 지점정보·연중일만으로 예측합니다")

# 물리적으로 불가능한 값 차단. 결측 제출은 실격이므로 NaN 도 막는다.
# 대체값은 예측 자체의 중앙값이라 외부 자료가 아니다.
ta = np.clip(np.nan_to_num(ta, nan=float(np.nanmedian(ta))), -30, 50)
hm = np.clip(np.nan_to_num(hm, nan=float(np.nanmedian(hm))), 0, 100)

pred = pd.DataFrame({
    "Date":   X["date"].dt.strftime("%Y%m%d").astype(int),
    "STN_ID": X["STN"].astype(int),
    "TA":     ta,
    "HM":     hm,
})
print(f"예측 완료 {len(pred)}행  TA {pred.TA.min():.1f}~{pred.TA.max():.1f}  "
      f"HM {pred.HM.min():.1f}~{pred.HM.max():.1f}")


## 3. 제출 파일 생성 — 수정하지 마세요

In [ ]:
# ── PRED_DATES 표현 점검 ─────────────────────────────────────
# 참가자가 자유 구현 중 PRED_DATES를 date/UTC Timestamp 등으로 바꿔도
# 최종 제출 행 검증은 PRED_START/PRED_END에서 독립적으로 수행합니다.
# 따라서 여기서는 중단하지 않고 경고만 합니다.
_bad_times = []
for _d in PRED_DATES:
    try:
        _ts = pd.Timestamp(_d)
        if _ts.tzinfo is None:
            _kst = _ts
        else:
            _kst = _ts.tz_convert("Asia/Seoul").tz_localize(None)

        if _kst.hour != EVAL_HOUR or _kst.minute != EVAL_MINUTE:
            _bad_times.append(_d)
    except Exception:
        _bad_times.append(_d)

if _bad_times:
    print(
        "[경고] 현재 PRED_DATES에 14:00 KST가 아닌 표현이 포함되어 있습니다. "
        "최종 제출 날짜/행 수는 PRED_START/PRED_END에서 독립 검증합니다. "
        f"예시: {_bad_times[:3]}"
    )

# ── KMA API 감사 로그 저장 ───────────────────────────────────
_audit_out = "/kaggle/working/api_audit_log.csv"
if "API_AUDIT_LOG" in globals():
    _audit_df = pd.DataFrame(
        API_AUDIT_LOG,
        columns=["method", "host", "path", "date", "sDate", "eDate"],
    )
    _audit_df.to_csv(_audit_out, index=False)

    if len(_audit_df):
        _non_le1b = _audit_df[
            ~_audit_df["path"].fillna("").str.contains("/LE1B/", regex=False)
        ]
        if len(_non_le1b):
            print(
                f"[경고] /LE1B/ 이외 KMA API 경로 호출이 {len(_non_le1b)}건 기록되었습니다. "
                f"감사 로그: {_audit_out}"
            )
        else:
            print(f"API 감사 로그 저장: {_audit_out} ({len(_audit_df)}건)")
    else:
        print(f"API 감사 로그 저장: {_audit_out} (기록 0건)")

# ── pred 검증 ─────────────────────────────────────────────
if "pred" not in dir():
    sys.exit("자유 구현 영역에서 'pred' DataFrame을 만들어야 합니다.")
if not isinstance(pred, pd.DataFrame):
    sys.exit(f"pred 는 DataFrame 이어야 합니다 (현재: {type(pred).__name__})")

need = {"Date", "STN_ID", "TA", "HM"}
if not need <= set(pred.columns):
    sys.exit(f"pred 컬럼 부족: {sorted(need - set(pred.columns))}")

# 날짜 × 지점의 모든 조합이 정확히 한 번씩 있어야 합니다
# 기대 날짜 × 지점 조합은 자유 구현 영역에서 바뀔 수 있는 PRED_DATES가 아니라
# 운영진 설정값 PRED_START/PRED_END에서 독립적으로 재계산합니다.
_ref_dates = []
_ref_d = datetime.strptime(PRED_START, "%Y%m%d")
_ref_end = datetime.strptime(PRED_END, "%Y%m%d")
while _ref_d <= _ref_end:
    _ref_dates.append(_ref_d)
    _ref_d += timedelta(days=1)

_want = {
    (int(d.strftime("%Y%m%d")), s)
    for d in _ref_dates
    for s in STATIONS
}
_got  = [(int(a), int(b)) for a, b in
         zip(pred["Date"].astype(int), pred["STN_ID"].astype(int))]
if len(_got) != len(set(_got)):
    sys.exit("pred 에 중복된 (Date, STN_ID) 조합이 있습니다.")
if set(_got) != _want:
    miss, extra = _want - set(_got), set(_got) - _want
    sys.exit(f"pred 행 구성 오류 — 누락 {len(miss)}개, 불필요 {len(extra)}개\n"
             f"  누락 예시: {sorted(miss)[:3]}\n"
             f"  불필요 예시: {sorted(extra)[:3]}")

# ── 제출 파일 구성 ────────────────────────────────────────
submission = pd.DataFrame({
    "ID": pred["Date"].astype(int).astype(str) + "_"
          + pred["STN_ID"].astype(int).astype(str),
    "TA": np.clip(pd.to_numeric(pred["TA"], errors="coerce"), -50, 50).round(2),
    "HM": np.clip(pd.to_numeric(pred["HM"], errors="coerce"), 0, 100).round(2),
}).sort_values("ID").reset_index(drop=True)

if submission[["TA", "HM"]].isna().any().any():
    n = int(submission[["TA", "HM"]].isna().any(axis=1).sum())
    sys.exit(f"결측 예측값 {n}행 — 모든 행에 값이 있어야 합니다.")

out = "/kaggle/working/submission.csv"
submission.to_csv(out, index=False)

print("=" * 52)
print(f"  저장 완료: {out}")
print(f"  {len(submission)}행 = {len(STATIONS)}지점 x {len(_ref_dates)}일")
print(f"  TA {submission.TA.min():.1f} ~ {submission.TA.max():.1f} C")
print(f"  HM {submission.HM.min():.1f} ~ {submission.HM.max():.1f} %")
print("=" * 52)
print(submission.head().to_string(index=False))

## 제출 전 점검

- [ ] 자유 구현 영역에 **날짜가 하드코딩되어 있지 않은가** (`PRED_DATES` 참조 확인)
- [ ] `PRED_DATES`의 예측 대상 시각이 매일 **14:00 KST**인가
- [ ] 사용하는 모든 위성 관측시각 `obs_dt`가 해당 예측의 **`target_dt` 이하**인가
- [ ] 과거 날짜 위성영상을 사용하더라도 평가일 14:00 KST 이후의 **미래 영상은 사용하지 않는가**
- [ ] GK-2A LE1B API의 `date`가 **KST가 아니라 UTC**로 변환되어 요청되는가
- [ ] 예: **14:00 KST → 05:00 UTC (`...0500`)**
- [ ] naive `obs_dt`는 KST로 해석된다는 점을 확인했는가 (이미 UTC인 naive datetime을 넣지 않았는가)
- [ ] 시간 피처가 있다면 **학습 때 정의한 시간 기준과 동일한가** (KST 기준 학습이면 14, UTC 기준 학습이면 5)
- [ ] 자유 구현 영역에 **학습 코드가 남아 있지 않은가**
- [ ] 평가기간 위성자료로 모델을 추가 학습·파인튜닝하고 있지 않은가
- [ ] 위성 다운로드 실패를 무시한 채 전부 대체값으로 예측하고 있지는 않은가
- [ ] 모델 Dataset이 Public이거나 운영진에 공유되었는가
- [ ] API 키를 코드에 직접 입력했는가 (Secrets 미사용)
- [ ] 실행 로그에 표시된 `station_list.csv`가 공식 96지점 파일인지 확인했는가
- [ ] `/kaggle/working/api_audit_log.csv`에 예상한 KMA API 경로/시각만 기록되었는가
- [ ] 설정 셀과 제출 검증 셀의 핵심 규칙을 임의 변경하지 않았는가

### 날짜 교체 리허설

`PRED_START` / `PRED_END` 를 **다른 기간으로 바꿔** Run All 해 보세요.  
운영진 채점과 똑같은 상황입니다.

예측 대상 시각은 매일 **14:00 KST**여야 합니다.  
위성 API 요청 시각은 사용하는 `obs_dt`에 따라 달라지며, `to_api_datetime(obs_dt, target_dt)`가 **UTC**로 변환합니다.

예를 들어:

```text
2026-08-24 14:00 KST -> API date 202608240500 UTC
2026-08-24 13:00 KST -> API date 202608240400 UTC
2026-08-23 23:00 KST -> API date 202608231400 UTC
```

모든 경우 `obs_dt <= target_dt`를 만족해야 합니다.

### 재현성 확인

Copy & Edit 로 사본을 만들어 그대로 Run All 했을 때 `submission.csv` 가  
**동일한 값**으로 나오는지 확인하세요. 달라진다면 무작위성이 남아 있는 것입니다.

### 동결 및 제출

1. **Save Version → "Save & Run All (Commit)"** ("Quick Save"는 동결로 인정되지 않습니다)
2. **Versions** 탭에서 버전 번호·저장 시각 확인 (UTC 표시, KST = UTC + 9h)
3. **Share** 에서 운영진 계정을 Collaborator로 추가